In [ ]:
import os
import sys
import random
import argparse
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from timm.layers import trunc_normal_
import json

from collections import defaultdict
# 设置中文显示
plt.rcParams["font.family"] = ["SimHei", "WenQuanYi Micro Hei", "Heiti TC"]
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
# ------------------------------
# 辅助函数
# ------------------------------
def extract_slice_number(sample_id):
    """从样本编号中提取_slice_后的数字（用于数值排序）"""
    try:
        return int(sample_id.split('_slice_')[1])
    except (IndexError, ValueError):
        return 0


def set_seed(seed):
    """设置全局随机种子，保证实验可复现"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"🌱 已设置全局随机种子：{seed}")


In [ ]:
# ------------------------------
# 1. 损失函数定义
# ------------------------------
def build_loss(args, class_weights=None, device='cpu'):
    """构建辅助分类损失；CRF损失在 StratigraphyCRFNet 内计算。"""
    weight = None

    if class_weights is not None:
        weight = torch.as_tensor(
            class_weights,
            dtype=torch.float32,
            device=device
        )

    if args.loss == "cross_entropy":
        return nn.CrossEntropyLoss(
            weight=weight,
            label_smoothing=args.label_smoothing
        )

    if args.loss == "focal":
        return FocalLoss(
            gamma=args.focal_gamma,
            class_weights=weight
        )

    raise ValueError(f"不支持的损失函数类型：{args.loss}")


class FocalLoss(nn.Module):
    def __init__(
        self,
        gamma=2.0,
        class_weights=None,
        reduction="mean"
    ):
        super().__init__()

        self.gamma = gamma
        self.reduction = reduction

        # 注册为buffer，可自动跟随模型/device迁移
        if class_weights is None:
            self.register_buffer("class_weights", None)
        else:
            self.register_buffer(
                "class_weights",
                class_weights.detach().clone()
            )

    def forward(self, logits, target):
        """logits: [N, num_classes], target: [N]"""
        log_probs = F.log_softmax(logits, dim=-1)

        log_pt = log_probs.gather(
            dim=1,
            index=target.unsqueeze(1)
        ).squeeze(1)

        pt = log_pt.exp()

        loss = -((1.0 - pt) ** self.gamma) * log_pt

        if self.class_weights is not None:
            loss = loss * self.class_weights[target]

        if self.reduction == "sum":
            return loss.sum()

        if self.reduction == "none":
            return loss

        return loss.mean()


In [ ]:
# ------------------------------
# 2. 数据预处理函数
# ------------------------------
def fill_nan_value(train_set, val_set, test_set):
    def to_numeric(arr):
        if arr is None:
            return None
        arr = np.array(arr, dtype=np.float64)
        arr[np.isinf(arr)] = np.nan
        return arr

    train_set = to_numeric(train_set)
    val_set = to_numeric(val_set)
    test_set = to_numeric(test_set)

    col_means = None
    if train_set is not None:
        col_means = np.nanmean(train_set, axis=0)
        col_means[np.isnan(col_means)] = 0.0
        ind_train = np.where(np.isnan(train_set))
        train_set[ind_train] = np.take(col_means, ind_train[1])

    if val_set is not None and col_means is not None:
        ind_val = np.where(np.isnan(val_set))
        val_set[ind_val] = np.take(col_means, ind_val[1])
    if test_set is not None and col_means is not None:
        ind_test = np.where(np.isnan(test_set))
        test_set[ind_test] = np.take(col_means, ind_test[1])

    return train_set, val_set, test_set


def normalize_train_val_test(train, val, test):
    if train is None:
        return val, test

    min_val = np.min(train, axis=0)
    max_val = np.max(train, axis=0)
    range_val = max_val - min_val
    range_val[range_val == 0] = 1.0  

    train_norm = (train - min_val) / range_val
    val_norm = (val - min_val) / range_val if val is not None else None
    test_norm = (test - min_val) / range_val if test is not None else None

    return train_norm, val_norm, test_norm


In [ ]:
# ------------------------------
# 3. 自定义数据集
# ------------------------------
class LogDataset(Dataset):
    def __init__(self, data, labels, well_names, sample_ids):
        self.data = data
        self.labels = labels
        self.well_names = well_names
        self.sample_ids = sample_ids

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx], self.well_names[idx], self.sample_ids[idx]


In [ ]:
# ------------------------------
# 4. 加载数据：固定测试集 + 剩余井8:2随机划分训练/验证
# ------------------------------
def load_single_dir(data_dir, seq_length, feature_cols):
    """加载单个文件夹下的所有井数据"""
    well_data_list = []
    all_labels = []

    well_files = [f for f in os.listdir(data_dir) if f.endswith(('.xlsx', '.xls'))]
    if not well_files:
        raise FileNotFoundError(f"数据目录 {data_dir} 中未找到Excel文件")

    for file in well_files:
        well_name = os.path.splitext(file)[0]
        file_path = os.path.join(data_dir, file)
        try:
            df = pd.read_excel(file_path)
            print(f"📊 正在处理井：{well_name}（文件：{file}，数据行数：{len(df)}）")

            for col in feature_cols:
                df[col] = pd.to_numeric(df[col], errors='coerce')

            well_sequences = []
            well_labels = []
            well_names = []
            sample_ids = []

            sample_id_groups = df.groupby('样本编号_1')

            sorted_sample_ids = sorted(
                sample_id_groups.groups.keys(),
                key=extract_slice_number
            )

            for sample_id in sorted_sample_ids:
                group = sample_id_groups.get_group(sample_id)
                features = group[feature_cols].values
                if len(features) == seq_length:
                    well_sequences.append(features)
                    label = group['切片地层标签_1'].iloc[0]
                    well_labels.append(label)
                    well_names.append(well_name)
                    sample_ids.append(sample_id)
                else:
                    print(f"❌ 井 {well_name} 样本 {sample_id} 长度为{len(features)}（需{seq_length}），已跳过")

            if len(well_sequences) == 0:
                print(f"⚠️  井 {well_name} 无有效序列，已跳过")
                continue

            well_data_list.append({
                'well_name': well_name,
                'sequences': well_sequences,
                'labels': well_labels,
                'well_names': well_names,
                'sample_ids': sample_ids,
                'sample_count': len(well_sequences)
            })
            all_labels.extend(well_labels)

            print(f"✅ 井 {well_name} 样本排序完成，顺序：{sample_ids[:5]}...（共{len(sample_ids)}个样本）")

        except Exception as e:
            print(f"❌ 处理井 {well_name} 失败：{str(e)}，已跳过")
            continue

    if not well_data_list:
        raise ValueError(f"❌ 目录 {data_dir} 中无有效井数据加载成功")

    return well_data_list, all_labels


def load_log_data_train_val_test(data_dir, target_test_wells, train_ratio=0.8, seq_length=120, random_seed=42):
    """
    加载数据：指定井作为测试集，其余井按 train_ratio 随机划分训练集和验证集。
    target_test_wells: list, 固定作为测试集的井名列表。
    train_ratio: float, 剩余井中用于训练集的比例；其余作为验证集。
    """
    feature_cols = [
        'GR_INPEFA',
        'GR', 'CNL', 'AC', 'CAL', 'DEN', 'GR_A2', 'GR_D1', 'DEN_A1', 
        'CNL_A3', 'CNL_D3', 'AC_A1', '向量维度_1', '向量维度_2', '向量维度_3', '向量维度_4', '向量维度_5'
    ]

    print(f"\n=== 加载所有数据（目录：{data_dir}）===")
    all_well_data, _ = load_single_dir(data_dir, seq_length, feature_cols)

    rng = random.Random(random_seed)

    test_well_data = []
    train_val_well_data = []
    target_test_wells = set(target_test_wells)

    for well in all_well_data:
        if well['well_name'] in target_test_wells:
            test_well_data.append(well)
        else:
            train_val_well_data.append(well)

    if len(test_well_data) == 0:
        raise ValueError("指定测试井没有匹配到任何有效井数据，请检查 target_test_wells")
    if len(train_val_well_data) < 2:
        raise ValueError("除测试井外的剩余井数量不足，无法继续划分训练集和验证集")

    rng.shuffle(train_val_well_data)
    train_count = int(len(train_val_well_data) * train_ratio)
    train_count = max(1, min(train_count, len(train_val_well_data) - 1))

    train_well_data = train_val_well_data[:train_count]
    val_well_data = train_val_well_data[train_count:]

    total_wells = len(all_well_data)
    print(
        f"\n📊 井数量统计：总井数={total_wells}, "
        f"训练井数={len(train_well_data)}, 验证井数={len(val_well_data)}, 测试井数={len(test_well_data)}"
    )

    all_labels = []
    for well in train_well_data + val_well_data + test_well_data:
        all_labels.extend(well['labels'])

    label_counts = pd.Series(all_labels).value_counts().sort_index()
    print(f"\n📈 类别数量统计：")
    for label, count in label_counts.items():
        print(f"  类别 '{label}'：{count} 个样本")

    def merge_wells_for_crf(wells, seq_slices=5):
        all_seq_blocks = []
        all_label_blocks = []
        all_well_names_blocks = []
        all_sample_ids_blocks = []

        for well in wells:
            seqs = np.array(well['sequences']).transpose(0, 2, 1)  # [N, C, T]
            labels = np.array(well['labels'])
            well_names = np.array(well['well_names'])
            sample_ids = np.array(well['sample_ids'])

            for i in range(len(seqs) - seq_slices + 1):
                all_seq_blocks.append(seqs[i:i+seq_slices])        # [S, C, T]
                all_label_blocks.append(labels[i:i+seq_slices])    # [S]
                all_well_names_blocks.append(well_names[i:i+seq_slices])
                all_sample_ids_blocks.append(sample_ids[i:i+seq_slices])

        return np.array(all_seq_blocks), np.array(all_label_blocks), np.array(all_well_names_blocks), np.array(all_sample_ids_blocks)

    X_train, y_train, train_well_names, train_sample_ids = merge_wells_for_crf(train_well_data)
    X_val, y_val, val_well_names, val_sample_ids = merge_wells_for_crf(val_well_data)
    X_test, y_test, test_well_names, test_sample_ids = merge_wells_for_crf(test_well_data)

    train_well_names_list = [well['well_name'] for well in train_well_data]
    val_well_names_list = [well['well_name'] for well in val_well_data]
    test_well_names_list = [well['well_name'] for well in test_well_data]
    total_samples = len(X_train) + len(X_val) + len(X_test)

    print(f"\n⚖️  数据集划分（固定测试集 + 剩余井随机8:2训练/验证）：")
    print(f"  总样本数：{total_samples}")
    print(f"  训练集：{len(train_well_data)} 口井，{len(X_train)} 个样本（{len(X_train) / total_samples * 100:.1f}%）")
    print(f"    训练井：{train_well_names_list}")
    print(f"  验证集：{len(val_well_data)} 口井，{len(X_val)} 个样本（{len(X_val) / total_samples * 100:.1f}%）")
    print(f"    验证井：{val_well_names_list}")
    print(f"  测试集：{len(test_well_data)} 口井，{len(X_test)} 个样本（{len(X_test) / total_samples * 100:.1f}%）")
    print(f"    测试井：{test_well_names_list}")

    le = LabelEncoder()
    le.fit(np.concatenate([y_train.flatten(), y_val.flatten(), y_test.flatten()]))

    y_train_encoded = le.transform(y_train.flatten()).reshape(y_train.shape)
    y_val_encoded = le.transform(y_val.flatten()).reshape(y_val.shape)
    y_test_encoded = le.transform(y_test.flatten()).reshape(y_test.shape)

    num_classes = len(le.classes_)
    print(f"\n🔤 标签编码：")
    print(f"  类别数：{num_classes}，映射关系：{dict(zip(le.classes_, range(num_classes)))}")

    return (X_train, y_train_encoded, train_well_names, train_sample_ids,
            X_val, y_val_encoded, val_well_names, val_sample_ids,
            X_test, y_test_encoded, test_well_names, test_sample_ids,
            num_classes, le, label_counts, train_well_data, val_well_data, test_well_data, random_seed)


In [ ]:
# ------------------------------
# 5. 详细评估函数（整井CRF解码）
# ------------------------------
def detailed_evaluate_model(
    dataloader,
    model,
    aux_loss_fn,
    device,
    args,
    label_encoder,
    eval_epoch,
):
    model.eval()

    total_loss = 0.0
    total_crf_loss = 0.0
    total_aux_loss = 0.0
    total_moe_loss = 0.0
    total_tokens = 0

    # 保存每个唯一切片的发射分数
    emission_records = {}

    with torch.no_grad():
        for (
            x_seq,
            y_seq,
            well_names_seq,
            sample_ids_seq
        ) in dataloader:

            x_seq = x_seq.to(
                device,
                non_blocking=True
            )

            y_seq = y_seq.to(
                device,
                non_blocking=True
            )

            crf_loss, moe_loss, emissions = model(
                x_seq,
                labels=y_seq,
                num_epoch_i=eval_epoch,
                warm_up_epoch=args.warm_up_epoch,
                return_emissions=True
            )

            aux_loss = aux_loss_fn(
                emissions.reshape(
                    -1,
                    args.num_class
                ),
                y_seq.reshape(-1)
            )

            batch_loss = (
                crf_loss
                + args.aux_loss_rate * aux_loss
                + args.moe_loss_rate * moe_loss
            )

            num_tokens = y_seq.numel()

            total_loss += (
                batch_loss.item() * num_tokens
            )

            total_crf_loss += (
                crf_loss.item() * num_tokens
            )

            total_aux_loss += (
                aux_loss.item() * num_tokens
            )

            total_moe_loss += (
                moe_loss.item() * num_tokens
            )

            total_tokens += num_tokens

            # 默认collate后，元数据是按序列位置排列的
            sample_ids_by_batch = list(
                zip(*sample_ids_seq)
            )

            well_names_by_batch = list(
                zip(*well_names_seq)
            )

            # 收集每个唯一切片的发射分数
            for b in range(x_seq.size(0)):
                for s in range(x_seq.size(1)):
                    sample_id = (
                        sample_ids_by_batch[b][s]
                    )

                    well_name = (
                        well_names_by_batch[b][s]
                    )

                    true_label = int(
                        y_seq[b, s].item()
                    )

                    current_emission = (
                        emissions[b, s]
                        .detach()
                        .float()
                        .cpu()
                    )

                    if sample_id not in emission_records:
                        emission_records[sample_id] = {
                            "well_name": well_name,
                            "true": true_label,
                            "slice_number":
                                extract_slice_number(
                                    sample_id
                                ),
                            "emission_sum":
                                current_emission.clone(),
                            "count": 1
                        }

                    else:
                        record = emission_records[
                            sample_id
                        ]

                        # 同一切片在不同窗口中的标签必须完全一致
                        if record["true"] != true_label:
                            raise ValueError(
                                f"同一样本标签不一致："
                                f"{sample_id}"
                            )

                        record[
                            "emission_sum"
                        ] += current_emission

                        record["count"] += 1

    # 按井排序后进行整井CRF解码
    well_to_samples = defaultdict(list)

    for sample_id, record in emission_records.items():
        well_to_samples[
            record["well_name"]
        ].append(
            (sample_id, record)
        )

    decoded_rows = []

    for well_name, well_items in (
        well_to_samples.items()
    ):
        # 严格按slice编号排序
        well_items = sorted(
            well_items,
            key=lambda item:
                item[1]["slice_number"]
        )

        # 对同一切片在多个重叠窗口中的emission取平均
        averaged_emissions = torch.stack([
            record["emission_sum"]
            / float(record["count"])
            for _, record in well_items
        ])

        # [1, well_length, num_classes]
        whole_well_emissions = (
            averaged_emissions
            .unsqueeze(0)
            .to(device)
        )

        # 整口井只执行一次CRF解码
        whole_well_predictions = model.decode(
            whole_well_emissions
        )[0].detach().cpu().tolist()

        for (
            sample_id,
            record
        ), pred_label in zip(
            well_items,
            whole_well_predictions
        ):
            decoded_rows.append({
                "sample_id": sample_id,
                "well_name": well_name,
                "true": record["true"],
                "pred": int(pred_label),
                "slice_number":
                    record["slice_number"]
            })

    df_clean = pd.DataFrame(decoded_rows)

    # 保证最终报告顺序稳定
    df_clean = df_clean.sort_values(
        ["well_name", "slice_number"]
    ).reset_index(drop=True)

    all_labels = df_clean[
        "true"
    ].to_numpy()

    all_preds = df_clean[
        "pred"
    ].to_numpy()

    avg_loss = (
        total_loss / max(total_tokens, 1)
    )

    avg_crf_loss = (
        total_crf_loss / max(total_tokens, 1)
    )

    avg_aux_loss = (
        total_aux_loss / max(total_tokens, 1)
    )

    avg_moe_loss = (
        total_moe_loss / max(total_tokens, 1)
    )

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    (
        precision_macro,
        recall_macro,
        f1_macro,
        _
    ) = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    (
        precision_micro,
        recall_micro,
        f1_micro,
        _
    ) = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="micro",
        zero_division=0
    )

    class_report = classification_report(
        all_labels,
        all_preds,
        target_names=label_encoder.classes_,
        zero_division=0
    )

    conf_matrix = confusion_matrix(
        all_labels,
        all_preds
    )

    well_acc_dict = {}

    for well_name, group in df_clean.groupby(
        "well_name"
    ):
        sorted_group = group.sort_values(
            "slice_number"
        ).reset_index(drop=True)

        well_acc = accuracy_score(
            sorted_group["true"],
            sorted_group["pred"]
        )

        well_acc_dict[well_name] = {
            "accuracy": well_acc,
            "sample_count": len(sorted_group),
            "correct_count": (
                sorted_group["true"]
                == sorted_group["pred"]
            ).sum(),
            "incorrect_count": (
                sorted_group["true"]
                != sorted_group["pred"]
            ).sum(),
            "sorted_samples": sorted_group[[
                "sample_id",
                "true",
                "pred",
                "slice_number"
            ]]
        }

    return {
        "avg_loss": avg_loss,
        "avg_crf_loss": avg_crf_loss,
        "avg_aux_loss": avg_aux_loss,
        "avg_moe_loss": avg_moe_loss,

        "overall_accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
        "class_report": class_report,
        "confusion_matrix": conf_matrix,
        "well_accuracy": well_acc_dict,

        "raw_data": (
            df_clean["well_name"].values,
            all_labels,
            all_preds,
            df_clean["sample_id"].values
        )
    }


# 层序错误统计函数
def count_stratigraphic_violations(
    well_accuracy_dict
):
    """
    统计预测结果中违反正常地层顺序的转移。

    标签编码：
        0：嘉一
        1：嘉三
        2：嘉二
        3：嘉五
        4：嘉四

    slice编号递增时的正常顺序：
        嘉五 -> 嘉四 -> 嘉三 -> 嘉二 -> 嘉一
    """
    order_rank = {
        3: 0,  # 嘉五
        4: 1,  # 嘉四
        1: 2,  # 嘉三
        2: 3,  # 嘉二
        0: 4   # 嘉一
    }

    reverse_count = 0
    skip_count = 0
    total_transition_count = 0

    violation_details = []

    for well_name, well_info in (
        well_accuracy_dict.items()
    ):
        samples = well_info[
            "sorted_samples"
        ].sort_values(
            "slice_number"
        ).reset_index(drop=True)

        predictions = samples[
            "pred"
        ].astype(int).tolist()

        sample_ids = samples[
            "sample_id"
        ].tolist()

        for index, (
            current_tag,
            next_tag
        ) in enumerate(
            zip(
                predictions[:-1],
                predictions[1:]
            )
        ):
            current_rank = order_rank[
                current_tag
            ]

            next_rank = order_rank[
                next_tag
            ]

            rank_difference = (
                next_rank - current_rank
            )

            total_transition_count += 1

            violation_type = None

            if rank_difference < 0:
                reverse_count += 1
                violation_type = "逆向转移"

            elif rank_difference > 1:
                skip_count += 1
                violation_type = "跨层跳跃"

            if violation_type is not None:
                violation_details.append({
                    "well_name": well_name,
                    "from_sample":
                        sample_ids[index],
                    "to_sample":
                        sample_ids[index + 1],
                    "from_tag":
                        current_tag,
                    "to_tag":
                        next_tag,
                    "type":
                        violation_type
                })

    return {
        "total_transitions":
            total_transition_count,

        "reverse_transitions":
            reverse_count,

        "skip_transitions":
            skip_count,

        "total_violations":
            reverse_count + skip_count,

        "violation_rate": (
            (
                reverse_count + skip_count
            ) / total_transition_count
            if total_transition_count > 0
            else 0.0
        ),

        "details":
            violation_details
    }


In [ ]:
# ------------------------------
# 6. 自定义结果保存函数
# ------------------------------
def save_cls_result(args, eval_results, train_time, label_encoder, save_path, random_seed):
    """保存完整训练结果到CSV"""
    result_df = pd.DataFrame({
        '时间': [time.strftime('%Y-%m-%d %H:%M:%S')],
        '随机种子': [random_seed],
        '数据目录': [args.data_dir],
        '序列长度': [args.seq_length],
        '嵌入维度': [args.emb_dim],
        '模型层数': [args.depth],
        '稀疏率': [args.sparse_rate],
        'Shapelet窗口': [args.shape_size],
        'Shapelet步长': [args.shape_stride],
        'MoE专家数': [args.moe_num_experts],
        'MoE Top-k': [args.moe_top_k],
        '稀疏预热轮数': [args.warm_up_epoch],
        '稀疏渐进轮数': [args.sparse_ramp_epoch],
        'Transformer层数': [args.transformer_layers],
        'Transformer头数': [args.transformer_nhead],
        '学习率': [args.lr],
        '权重衰减': [args.weight_decay],
        '批次大小': [args.batch_size],
        '训练轮数': [args.epoch],
        '整体准确率': [eval_results['overall_accuracy']],
        '宏平均精确率': [eval_results['precision_macro']],
        '宏平均召回率': [eval_results['recall_macro']],
        '宏平均F1': [eval_results['f1_macro']],
        '微平均精确率': [eval_results['precision_micro']],
        '微平均召回率': [eval_results['recall_micro']],
        '微平均F1': [eval_results['f1_micro']],
        '训练时间(分钟)': [train_time / 60],
        '损失函数': [args.loss],
        '优化器': [args.optimizer],
        'MoE损失权重': [args.moe_loss_rate]
    })

    if os.path.exists(save_path):
        result_df.to_csv(save_path, mode='a', header=False, index=False, encoding='utf-8-sig')
    else:
        result_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"📁 训练结果已保存至：{save_path}")


In [ ]:
# ------------------------------
# 7. 主训练流程
# ------------------------------
if __name__ == '__main__':
    parser = argparse.ArgumentParser(description="SoftShapeTransformerNet（三路径混合模型）- 测井数据分类")

    # 基础设置
    parser.add_argument('--random_seed', type=int, default=0, help='随机种子（默认0）')
    parser.add_argument('--data_dir', type=str,
                        default="./data",
                        help='数据目录（包含所有井数据的文件夹）')
    parser.add_argument('--seq_length', type=int, default=120, help='时序序列固定长度')
    parser.add_argument('--train_ratio', type=float, default=0.8, help='剩余井训练集比例（默认0.8）')

    # 模型设置
    parser.add_argument('--emb_dim', type=int, default=128, help='Shapelet嵌入维度')
    parser.add_argument('--depth', type=int, default=2, help='SoftShapeNet层数')
    parser.add_argument('--sparse_rate', type=float, default=0.3, help='最终目标稀疏率')
    parser.add_argument('--shape_size', type=int, default=60, help='Shapelet窗口大小（10-30）')
    parser.add_argument('--shape_stride', type=int, default=5, help='Shapelet滑动步长（5-15）')
    parser.add_argument('--moe_num_experts', type=int, default=4, help='MoE专家数')
    parser.add_argument('--moe_top_k', type=int, default=2, help='每个token选择的专家数')
    parser.add_argument('--transformer_nhead', type=int, default=4, help='Transformer注意力头数')
    parser.add_argument('--transformer_layers', type=int, default=2, help='Transformer编码器层数')

    # 训练设置
    parser.add_argument('--warm_up_epoch', type=int, default=5, help='稀疏化预热轮数')
    parser.add_argument('--sparse_ramp_epoch', type=int, default=20, help='从0渐进到目标稀疏率的轮数')
    parser.add_argument('--moe_loss_rate', type=float, default=0.02, help='MoE可导均衡损失权重')
    parser.add_argument('--loss', type=str, default='cross_entropy', help='损失函数（cross_entropy/focal）')
    parser.add_argument('--optimizer', type=str, default='adam', help='优化器（adam/sgd）')
    parser.add_argument('--lr', type=float, default=0.001, help='初始学习率')
    parser.add_argument('--weight_decay', type=float, default=1e-5, help='权重衰减')
    parser.add_argument('--batch_size', type=int, default=32, help='批次大小')
    parser.add_argument('--epoch', type=int, default=200, help='训练轮数')
    parser.add_argument('--cuda', type=str, default='cuda:0', help='GPU设备（cuda:0/cpu）')

    parser.add_argument("--aux_loss_rate", type=float, default=0.3, help="加权CE/Focal辅助损失权重")

    parser.add_argument("--label_smoothing", type=float, default=0.05, help="交叉熵标签平滑系数")

    parser.add_argument("--focal_gamma", type=float, default=2.0, help="Focal Loss的gamma参数")

    # 结果设置
    parser.add_argument('--save_dir', type=str, default='./result_specified_split', help='结果保存路径')
    parser.add_argument('--save_csv_name', type=str, default='log_stratigraphy_random_split_result',
                        help='结果CSV文件名')
    parser.add_argument('--save_model_name', type=str, default='best_model_random_split.pth',
                        help='最佳模型文件名')

    if 'ipykernel' in sys.modules:
        args = parser.parse_args([])
    else:
        args, _ = parser.parse_known_args()

    # ============================================================
    # 固定配置：整井CRF + 4专家Top-k=2 + 渐进稀疏0.3
    # ============================================================
    args.experiment_name = "MoE4_Top2_Sparse03"
    args.transition_penalty = -3.0

    args.save_model_name = (
        f"best_model_{args.experiment_name}_"
        f"seed_{args.random_seed}.pth"
    )
    args.save_csv_name = (
        f"log_stratigraphy_{args.experiment_name}_"
        f"seed_{args.random_seed}"
    )

    print("=" * 60)
    print(f"当前实验：{args.experiment_name}")
    print(f"MoE专家数：{args.moe_num_experts}")
    print(f"MoE Top-k：{args.moe_top_k}")
    print(f"目标稀疏率：{args.sparse_rate}")
    print(f"预热轮数：{args.warm_up_epoch}")
    print(f"渐进轮数：{args.sparse_ramp_epoch}")
    print(f"MoE损失权重：{args.moe_loss_rate}")
    print(f"非法转移惩罚：{args.transition_penalty}")
    print("=" * 60)

    # 初始化环境
    os.makedirs(args.save_dir, exist_ok=True)
    device = torch.device(args.cuda if torch.cuda.is_available() else "cpu")
    print(f"=== 训练环境初始化 ===")
    print(f"设备：{device} | 结果路径：{args.save_dir} | 随机种子：{args.random_seed}")
    print(f"数据目录：{args.data_dir}")
    print(f"剩余井训练集比例：{args.train_ratio} | 剩余井验证集比例：{1 - args.train_ratio}")

    # 加载数据
    print("\n=== 加载测井数据 ===")
    target_test_wells = ['指定的测试集井']

    print(f"设置的固定测试集井为：{target_test_wells}")

    try:
        (X_train, y_train, train_well_names, train_sample_ids,
         X_val, y_val, val_well_names, val_sample_ids,
         X_test, y_test, test_well_names, test_sample_ids,
         num_classes, label_encoder, label_counts, train_wells, val_wells, test_wells, used_seed) = load_log_data_train_val_test(
            data_dir=args.data_dir,
            target_test_wells=target_test_wells,
            train_ratio=args.train_ratio,
            seq_length=args.seq_length,
            random_seed=args.random_seed
        )
    except Exception as e:
        print(f"❌ 数据加载失败：{str(e)}")
        sys.exit(1)

    # 设置全局随机种子
    set_seed(args.random_seed)
    start_time = time.time()

    args.num_class = num_classes
    args.num_channels = X_train.shape[2]  # 特征变量数
    args.seq_len = X_train.shape[3]       # 序列长度

    print(f"\n=== 数据维度 ===")
    print(f"训练集：{X_train.shape[0]} 序列块 × {X_train.shape[1]} 连续切片 × {args.num_channels} 变量 × {args.seq_len} 长度")
    print(f"验证集：{X_val.shape[0]} 序列块 × {X_val.shape[1]} 连续切片 × {args.num_channels} 变量 × {args.seq_len} 长度")
    print(f"测试集：{X_test.shape[0]} 序列块 × {X_test.shape[1]} 连续切片 × {args.num_channels} 变量 × {args.seq_len} 长度")

    # 数据预处理：先降维到3D做填充/归一化，再恢复4D
    print("\n=== 数据预处理 ===")
    B_tr, S, C, T_len = X_train.shape
    B_va = X_val.shape[0]
    B_te = X_test.shape[0]

    X_train_3d = X_train.reshape(-1, C, T_len)
    X_val_3d = X_val.reshape(-1, C, T_len)
    X_test_3d = X_test.reshape(-1, C, T_len)

    X_train_3d, X_val_3d, X_test_3d = fill_nan_value(X_train_3d, X_val_3d, X_test_3d)
    X_train_3d, X_val_3d, X_test_3d = normalize_train_val_test(X_train_3d, X_val_3d, X_test_3d)

    X_train = X_train_3d.reshape(B_tr, S, C, T_len)
    X_val = X_val_3d.reshape(B_va, S, C, T_len)
    X_test = X_test_3d.reshape(B_te, S, C, T_len)

    print("✅ 缺失值填充+归一化完成")

    flat_train_labels = y_train.reshape(-1)

    # 类别权重顺序严格对应类别索引0、1、2、3、4
    class_weights = compute_class_weight(class_weight="balanced", classes=np.arange(num_classes), y=flat_train_labels)

    # 归一化到均值为1，使辅助损失量级更稳定
    class_weights = class_weights / class_weights.mean()

    print("类别映射：")
    print(dict(enumerate(label_encoder.classes_)))

    print(
        "⚖️类别权重：",
        {
            label_encoder.classes_[i]: round(float(class_weights[i]), 4)
            for i in range(num_classes)
        }
    )

    expected_classes = ["嘉一", "嘉三", "嘉二", "嘉五", "嘉四"]

    assert list(label_encoder.classes_) == expected_classes, (
        f"标签编码不一致！\n"
        f"当前编码：{list(label_encoder.classes_)}\n"
        f"CRF约束编码：{expected_classes}"
    )

    # 构建数据集和DataLoader
    train_set = LogDataset(
        torch.from_numpy(X_train).float(),
        torch.from_numpy(y_train).long(),
        train_well_names.tolist(),
        train_sample_ids.tolist()
    )
    val_set = LogDataset(
        torch.from_numpy(X_val).float(),
        torch.from_numpy(y_val).long(),
        val_well_names.tolist(),
        val_sample_ids.tolist()
    )
    test_set = LogDataset(
        torch.from_numpy(X_test).float(),
        torch.from_numpy(y_test).long(),
        test_well_names.tolist(),
        test_sample_ids.tolist()
    )

    num_workers = 0 if sys.platform == 'win32' else 4
    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True, num_workers=num_workers,
                              pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=args.batch_size * 2, shuffle=False, num_workers=num_workers,
                            pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=args.batch_size * 2, shuffle=False, num_workers=num_workers,
                             pin_memory=True)
    print(f"\n=== 数据加载器 ===")
    print(f"训练集迭代次数：{len(train_loader)} | 验证集迭代次数：{len(val_loader)} | 测试集迭代次数：{len(test_loader)}")
    print(f"✅ 验证集和测试集DataLoader已禁用shuffle，保持样本顺序")

    # 初始化模型
    print("\n=== 初始化模型 ===")
    try:
        from models.SFSD import SoftShapeTransformerNet, StratigraphyCRFNet
    except ImportError as e:
        print(f"❌ 模型导入失败：{str(e)}")
        print("请确保 SFSD.py 在 models 目录下")
        sys.exit(1)

    base_model = SoftShapeTransformerNet(
        seq_len=args.seq_len,
        shape_size=args.shape_size,
        num_channels=args.num_channels,
        emb_dim=args.emb_dim,
        sparse_rate=args.sparse_rate,
        depth=args.depth,
        num_experts=args.moe_num_experts,
        num_classes=args.num_class,
        stride=args.shape_stride,
        transformer_nhead=args.transformer_nhead,
        transformer_layers=args.transformer_layers,
        moe_top_k=args.moe_top_k,
        sparse_ramp_epoch=args.sparse_ramp_epoch
    ).to(device)

    model = StratigraphyCRFNet(
        base_model=base_model,
        num_classes=args.num_class,
        transition_penalty=args.transition_penalty
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✅ 模型初始化完成：可训练参数 {total_params / 1e6:.2f} M")
    print(f"📊 模型架构：三路径混合模型（SoftShape MoE + 类别特定Transformer + 通用Transformer）")

    # 损失函数和优化器
    aux_loss_fn = build_loss(args, class_weights=class_weights, device=device)
    if args.optimizer == 'adam':
        optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    elif args.optimizer == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, weight_decay=args.weight_decay, momentum=0.9,
                                    nesterov=True)
    else:
        raise ValueError(f"不支持的优化器：{args.optimizer}")

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epoch, eta_min=1e-6)

    import copy

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    best_val_f1 = -float("inf")
    best_val_acc = 0.0
    best_epoch = 0
    best_model_state = None
    early_stop_counter = 0
    patience = 30

    print("\n=== 开始训练 ===")
    for epoch in range(args.epoch):
        model.train()
        train_loss = 0.0
        train_crf_loss = 0.0
        train_aux_loss = 0.0
        train_moe_loss = 0.0

        router_importance_sum = np.zeros(args.moe_num_experts, dtype=np.float64)
        router_selection_sum = np.zeros(args.moe_num_experts, dtype=np.float64)
        router_entropy_sum = 0.0
        router_stat_batches = 0
        model.base_model.moe.reset_routing_stats()

        correct = 0
        total = 0

        for x_seq, y_seq, _, _ in train_loader:
            x_seq = x_seq.to(device, non_blocking=True)
            y_seq = y_seq.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            crf_loss, moe_loss, emissions = model(
                x_seq,
                labels=y_seq,
                num_epoch_i=epoch,
                warm_up_epoch=args.warm_up_epoch,
                return_emissions=True
            )

            aux_loss = aux_loss_fn(
                emissions.reshape(-1, args.num_class),
                y_seq.reshape(-1)
            )

            batch_loss = (
                crf_loss
                + args.aux_loss_rate * aux_loss
                + args.moe_loss_rate * moe_loss
            )

            if not torch.isfinite(batch_loss):
                raise RuntimeError(
                    f"检测到非有限损失："
                    f"total={batch_loss.item()}, "
                    f"crf={crf_loss.item()}, "
                    f"aux={aux_loss.item()}, "
                    f"moe={moe_loss.item()}"
                )

            batch_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            batch_size = x_seq.size(0)

            train_loss += batch_loss.item() * batch_size
            train_crf_loss += crf_loss.item() * batch_size
            train_aux_loss += aux_loss.item() * batch_size
            train_moe_loss += moe_loss.item() * batch_size

            moe_module = model.base_model.moe
            if moe_module.last_importance is not None:
                router_importance_sum += moe_module.last_importance.cpu().numpy()
                router_selection_sum += moe_module.last_selection_share.cpu().numpy()
                router_entropy_sum += float(moe_module.last_router_entropy.cpu().item())
                router_stat_batches += 1

            # 直接解码本次前向传播得到的 emissions，无需再次执行模型
            with torch.no_grad():
                best_paths = model.decode(emissions.detach())

                correct += (best_paths == y_seq).sum().item()
                total += y_seq.numel()

        dataset_size = len(train_loader.dataset)

        avg_train_loss = train_loss / dataset_size
        avg_train_crf_loss = train_crf_loss / dataset_size
        avg_train_aux_loss = train_aux_loss / dataset_size
        avg_train_moe_loss = train_moe_loss / dataset_size

        train_acc = correct / max(total, 1)

        current_lr = optimizer.param_groups[0]['lr']

        # 验证阶段（整井CRF解码）
        val_eval = detailed_evaluate_model(
            val_loader,
            model,
            aux_loss_fn,
            device,
            args,
            label_encoder,
            eval_epoch=epoch
        )
        val_loss = val_eval['avg_loss']
        val_acc = val_eval['overall_accuracy']

        scheduler.step()
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        # 打印日志
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1:3d}/{args.epoch:3d} | LR: {current_lr:.6f}")
            print(
                f"训练集：总损失 {avg_train_loss:.4f} | "
                f"CRF {avg_train_crf_loss:.4f} | "
                f"辅助分类 {avg_train_aux_loss:.4f} | "
                f"MoE {avg_train_moe_loss:.4f} | "
                f"准确率 {train_acc:.4f}"
            )
            print(
                f"验证集：总损失 {val_eval['avg_loss']:.4f} | "
                f"CRF {val_eval['avg_crf_loss']:.4f} | "
                f"辅助分类 {val_eval['avg_aux_loss']:.4f} | "
                f"MoE {val_eval['avg_moe_loss']:.4f} | "
                f"准确率 {val_eval['overall_accuracy']:.4f} | "
                f"Macro-F1 {val_eval['f1_macro']:.4f}"
            )
            print(
                f"当前渐进稀疏率："
                f"{model.base_model.current_sparse_rate:.4f}"
            )
            if router_stat_batches > 0:
                avg_importance = router_importance_sum / router_stat_batches
                avg_selection = router_selection_sum / router_stat_batches
                avg_entropy = router_entropy_sum / router_stat_batches
                print("专家软概率：", np.round(avg_importance, 4).tolist())
                print("专家选择占比：", np.round(avg_selection, 4).tolist())
                print(f"归一化路由熵：{avg_entropy:.4f}")
            else:
                print("MoE尚未启用：仍处于预热阶段")
            print(f"  早停计数器：{early_stop_counter}/{patience} | 最佳准确率：{best_val_acc:.4f}")
            print("-" * 80)

        # 保存最佳模型（基于验证集Macro-F1）
        val_acc = val_eval["overall_accuracy"]
        val_f1 = val_eval["f1_macro"]

        moe_training_active = (
            epoch + 1 >= args.warm_up_epoch + 1
        )

        if not moe_training_active:
            early_stop_counter = 0
        elif val_f1 > best_val_f1 + 1e-4:
            best_val_f1 = val_f1
            best_val_acc = val_acc
            best_epoch = epoch + 1

            best_model_state = copy.deepcopy(model.state_dict())

            early_stop_counter = 0

            model_save_path = os.path.join(
                args.save_dir,
                args.save_model_name
            )

            torch.save(
                {
                    "epoch": best_epoch,
                    "model_state_dict": best_model_state,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_f1": best_val_f1,
                    "best_val_acc": best_val_acc,
                    "args": vars(args),
                    "used_seed": args.random_seed,
                    "class_weights": class_weights
                },
                model_save_path
            )

            print(
                f"保存最佳模型：Epoch {best_epoch} | "
                f"Macro-F1 {best_val_f1:.4f} | "
                f"准确率 {best_val_acc:.4f}"
            )
        else:
            early_stop_counter += 1

            if early_stop_counter >= patience:
                print(
                    f"早停触发：连续{patience}轮"
                    f"验证集Macro-F1未提升"
                )
                break

    # 训练后处理
    total_train_time = time.time() - start_time
    print(f"\n=== 训练完成 ===")
    print(f"总训练时间：{total_train_time:.2f} 秒（{total_train_time / 60:.2f} 分钟）")
    print(f"使用的随机种子：{args.random_seed}")
    print(f"最佳验证准确率：{best_val_acc:.4f}")

    # 加载最佳模型进行最终评估
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        final_eval = detailed_evaluate_model(
            val_loader,
            model,
            aux_loss_fn,
            device,
            args,
            label_encoder,
            # best_epoch是从1开始记录的
            eval_epoch=best_epoch - 1
        )
        test_eval = detailed_evaluate_model(
            test_loader,
            model,
            aux_loss_fn,
            device,
            args,
            label_encoder,
            eval_epoch=best_epoch - 1
        )
        print(f"\n=== 验证集与测试集最终准确率 ===")
        print(f"验证集：准确率 {final_eval['overall_accuracy']:.4f} | Macro-F1 {final_eval['f1_macro']:.4f}")
        print(f"测试集：准确率 {test_eval['overall_accuracy']:.4f} | Macro-F1 {test_eval['f1_macro']:.4f}")

        # 统计最终预测中的层序错误
        violation_stats = (
            count_stratigraphic_violations(
                final_eval["well_accuracy"]
            )
        )
        test_violation_stats = (
            count_stratigraphic_violations(
                test_eval["well_accuracy"]
            )
        )

        print("\n验证集层序错误统计：")
        print(
            f"预测转移总数："
            f"{violation_stats['total_transitions']}"
        )
        print(
            f"逆向转移数："
            f"{violation_stats['reverse_transitions']}"
        )
        print(
            f"跨层跳跃数："
            f"{violation_stats['skip_transitions']}"
        )
        print(
            f"层序错误总数："
            f"{violation_stats['total_violations']}"
        )
        print(
            f"层序错误率："
            f"{violation_stats['violation_rate']:.4f}"
        )

        print("\n测试集层序错误统计：")
        print(
            f"预测转移总数："
            f"{test_violation_stats['total_transitions']}"
        )
        print(
            f"逆向转移数："
            f"{test_violation_stats['reverse_transitions']}"
        )
        print(
            f"跨层跳跃数："
            f"{test_violation_stats['skip_transitions']}"
        )
        print(
            f"层序错误总数："
            f"{test_violation_stats['total_violations']}"
        )
        print(
            f"层序错误率："
            f"{test_violation_stats['violation_rate']:.4f}"
        )

        print(f"\n=== 最佳模型详细评估结果 ===")
        print(f"📊 整体指标：")
        print(f"  准确率：{final_eval['overall_accuracy']:.4f}")
        print(f"  宏平均精确率：{final_eval['precision_macro']:.4f}")
        print(f"  宏平均召回率：{final_eval['recall_macro']:.4f}")
        print(f"  宏平均F1：{final_eval['f1_macro']:.4f}")
        print(f"  微平均精确率：{final_eval['precision_micro']:.4f}")
        print(f"  微平均召回率：{final_eval['recall_micro']:.4f}")
        print(f"  微平均F1：{final_eval['f1_micro']:.4f}")

        print(f"\n🏭 各井验证准确率：")
        well_acc_df = pd.DataFrame.from_dict(final_eval['well_accuracy'], orient='index')
        well_acc_df = well_acc_df.sort_values('accuracy', ascending=False)
        well_acc_df['accuracy'] = well_acc_df['accuracy'].apply(lambda x: f"{x:.4f}")
        print(well_acc_df[['sample_count', 'correct_count', 'incorrect_count', 'accuracy']])

        print(f"\n📋 验证集分类详细报告：")
        print(final_eval['class_report'])

        print(f"\n🏭 各井测试准确率：")
        test_well_acc_df = pd.DataFrame.from_dict(test_eval['well_accuracy'], orient='index')
        test_well_acc_df = test_well_acc_df.sort_values('accuracy', ascending=False)
        test_well_acc_df['accuracy'] = test_well_acc_df['accuracy'].apply(lambda x: f"{x:.4f}")
        print(test_well_acc_df[['sample_count', 'correct_count', 'incorrect_count', 'accuracy']])

        print(f"\n📋 测试集分类详细报告：")
        print(test_eval['class_report'])

        # 打印每口验证集井每个样本的详细结果
        print(f"\n=== 验证集样本详细结果（按slice编号数字升序排列）===")
        for well_name, well_info in final_eval['well_accuracy'].items():
            print(f"\n📌 井：{well_name}（准确率：{well_info['accuracy']:.4f}，样本数：{well_info['sample_count']}）")
            print(f"{'样本编号':<20} {'实际标签':<15} {'预测标签':<15} {'是否正确'}")
            print("-" * 65)

            samples_df = well_info['sorted_samples'].copy()
            samples_df['true_label_name'] = samples_df['true'].map(lambda x: label_encoder.classes_[x])
            samples_df['pred_label_name'] = samples_df['pred'].map(lambda x: label_encoder.classes_[x])
            samples_df['is_correct'] = samples_df['true'] == samples_df['pred']

            for _, row in samples_df.iterrows():
                correct_mark = "✅" if row['is_correct'] else "❌"
                print(
                    f"{row['sample_id']:<20} {row['true_label_name']:<15} {row['pred_label_name']:<15} {correct_mark}")

        # 保存详细结果文本
        detail_save_path = os.path.join(
            args.save_dir,
            f"detailed_validation_report_"
            f"{args.experiment_name}_"
            f"seed_{args.random_seed}.txt"
        )
        with open(detail_save_path, 'w', encoding='utf-8') as f:
            f.write(f"实验名称：{args.experiment_name}\n")
            f.write(f"MoE专家数：{args.moe_num_experts}\n")
            f.write(f"MoE Top-k：{args.moe_top_k}\n")
            f.write(f"目标稀疏率：{args.sparse_rate}\n")
            f.write(f"稀疏预热轮数：{args.warm_up_epoch}\n")
            f.write(f"稀疏渐进轮数：{args.sparse_ramp_epoch}\n")
            f.write(f"MoE损失权重：{args.moe_loss_rate}\n")
            f.write(f"非法转移惩罚：{args.transition_penalty}\n")

            f.write("=== 测井数据分类验证详细报告（三路径混合模型）===\n")
            f.write(f"生成时间：{time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"使用的随机种子：{args.random_seed}\n")
            f.write(f"数据目录：{args.data_dir}\n")
            f.write(f"剩余井训练集比例：{args.train_ratio} | 剩余井验证集比例：{1 - args.train_ratio}\n")
            f.write(f"最佳模型Epoch：{best_epoch}\n")
            f.write(f"总训练时间：{total_train_time:.2f}秒\n\n")

            f.write("一、验证集整体分类指标\n")
            f.write(f"准确率：{final_eval['overall_accuracy']:.4f}\n")
            f.write(f"宏平均精确率：{final_eval['precision_macro']:.4f}\n")
            f.write(f"宏平均召回率：{final_eval['recall_macro']:.4f}\n")
            f.write(f"宏平均F1：{final_eval['f1_macro']:.4f}\n")
            f.write(f"微平均精确率：{final_eval['precision_micro']:.4f}\n")
            f.write(f"微平均召回率：{final_eval['recall_micro']:.4f}\n")
            f.write(f"微平均F1：{final_eval['f1_micro']:.4f}\n\n")

            f.write("二、测试集整体分类指标\n")
            f.write(f"准确率：{test_eval['overall_accuracy']:.4f}\n")
            f.write(f"宏平均精确率：{test_eval['precision_macro']:.4f}\n")
            f.write(f"宏平均召回率：{test_eval['recall_macro']:.4f}\n")
            f.write(f"宏平均F1：{test_eval['f1_macro']:.4f}\n")
            f.write(f"微平均精确率：{test_eval['precision_micro']:.4f}\n")
            f.write(f"微平均召回率：{test_eval['recall_micro']:.4f}\n")
            f.write(f"微平均F1：{test_eval['f1_micro']:.4f}\n\n")

            f.write("三、数据集信息\n")
            f.write(f"训练井：{[well['well_name'] for well in train_wells]}\n")
            f.write(f"验证井：{[well['well_name'] for well in val_wells]}\n")
            f.write(f"测试井：{[well['well_name'] for well in test_wells]}\n")
            f.write(f"训练集样本数：{len(X_train)} | 验证集样本数：{len(X_val)} | 测试集样本数：{len(X_test)}\n\n")

            f.write("四、验证集各井准确率统计\n")
            f.write(well_acc_df[['sample_count', 'correct_count', 'incorrect_count', 'accuracy']].to_string())
            f.write("\n\n五、验证集分类详细报告\n")
            f.write(final_eval['class_report'])

            f.write("\n\n六、测试集各井准确率统计\n")
            f.write(test_well_acc_df[['sample_count', 'correct_count', 'incorrect_count', 'accuracy']].to_string())
            f.write("\n\n七、测试集分类详细报告\n")
            f.write(test_eval['class_report'])

            f.write("\n\n八、验证集样本详细结果（按slice编号数字升序排列）\n")
            for well_name, well_info in final_eval['well_accuracy'].items():
                f.write(f"\n📌 井：{well_name}（准确率：{well_info['accuracy']:.4f}，样本数：{well_info['sample_count']}）\n")
                f.write(f"{'样本编号':<20} {'实际标签':<15} {'预测标签':<15} {'是否正确'}\n")
                f.write("-" * 65 + "\n")

                samples_df = well_info['sorted_samples'].copy()
                samples_df['true_label_name'] = samples_df['true'].map(lambda x: label_encoder.classes_[x])
                samples_df['pred_label_name'] = samples_df['pred'].map(lambda x: label_encoder.classes_[x])
                samples_df['is_correct'] = samples_df['true'] == samples_df['pred']

                for _, row in samples_df.iterrows():
                    correct_mark = "✅" if row['is_correct'] else "❌"
                    f.write(
                        f"{row['sample_id']:<20} {row['true_label_name']:<15} {row['pred_label_name']:<15} {correct_mark}\n")

            f.write("\n\n九、测试集样本详细结果（按slice编号数字升序排列）\n")
            for well_name, well_info in test_eval['well_accuracy'].items():
                f.write(f"\n📌 井：{well_name}（准确率：{well_info['accuracy']:.4f}，样本数：{well_info['sample_count']}）\n")
                f.write(f"{'样本编号':<20} {'实际标签':<15} {'预测标签':<15} {'是否正确'}\n")
                f.write("-" * 65 + "\n")

                samples_df = well_info['sorted_samples'].copy()
                samples_df['true_label_name'] = samples_df['true'].map(lambda x: label_encoder.classes_[x])
                samples_df['pred_label_name'] = samples_df['pred'].map(lambda x: label_encoder.classes_[x])
                samples_df['is_correct'] = samples_df['true'] == samples_df['pred']

                for _, row in samples_df.iterrows():
                    correct_mark = "✅" if row['is_correct'] else "❌"
                    f.write(
                        f"{row['sample_id']:<20} {row['true_label_name']:<15} {row['pred_label_name']:<15} {correct_mark}\n")

            # 十、验证集层序错误统计
            f.write("\n\n十、验证集层序错误统计\n")
            f.write(f"预测转移总数：{violation_stats['total_transitions']}\n")
            f.write(f"逆向转移数：{violation_stats['reverse_transitions']}\n")
            f.write(f"跨层跳跃数：{violation_stats['skip_transitions']}\n")
            f.write(f"层序错误总数：{violation_stats['total_violations']}\n")
            f.write(f"层序错误率：{violation_stats['violation_rate']:.4f}\n")
            f.write("\n层序错误详细位置：\n")
            f.write(
                f"{'井名':<15}"
                f"{'前一样本':<25}"
                f"{'后一样本':<25}"
                f"{'类型':<12}\n"
            )
            f.write("-" * 85 + "\n")

            for item in violation_stats["details"]:
                from_name = label_encoder.inverse_transform(
                    [item["from_tag"]]
                )[0]
                to_name = label_encoder.inverse_transform(
                    [item["to_tag"]]
                )[0]
                transition_text = (
                    f"{from_name}->{to_name}"
                )
                f.write(
                    f"{item['well_name']:<15}"
                    f"{item['from_sample']:<25}"
                    f"{item['to_sample']:<25}"
                    f"{item['type']:<12}"
                    f"{transition_text}\n"
                )

            f.write("\n\n十一、测试集层序错误统计\n")
            f.write(f"预测转移总数：{test_violation_stats['total_transitions']}\n")
            f.write(f"逆向转移数：{test_violation_stats['reverse_transitions']}\n")
            f.write(f"跨层跳跃数：{test_violation_stats['skip_transitions']}\n")
            f.write(f"层序错误总数：{test_violation_stats['total_violations']}\n")
            f.write(f"层序错误率：{test_violation_stats['violation_rate']:.4f}\n")
            f.write("\n层序错误详细位置：\n")
            f.write(
                f"{'井名':<15}"
                f"{'前一样本':<25}"
                f"{'后一样本':<25}"
                f"{'类型':<12}\n"
            )
            f.write("-" * 85 + "\n")

            for item in test_violation_stats["details"]:
                from_name = label_encoder.inverse_transform(
                    [item["from_tag"]]
                )[0]
                to_name = label_encoder.inverse_transform(
                    [item["to_tag"]]
                )[0]
                transition_text = (
                    f"{from_name}->{to_name}"
                )
                f.write(
                    f"{item['well_name']:<15}"
                    f"{item['from_sample']:<25}"
                    f"{item['to_sample']:<25}"
                    f"{item['type']:<12}"
                    f"{transition_text}\n"
                )

        print(f"\n📄 详细报告已保存至：{detail_save_path}")

        # 保存验证集混淆矩阵
        conf_matrix = final_eval['confusion_matrix']
        conf_df = pd.DataFrame(conf_matrix, index=label_encoder.classes_, columns=label_encoder.classes_)
        conf_csv_path = os.path.join(args.save_dir, f'confusion_matrix_seed_{args.random_seed}.csv')
        conf_df.to_csv(conf_csv_path, encoding='utf-8-sig')
        print(f"📊 混淆矩阵已保存至：{conf_csv_path}")

        # 验证集混淆矩阵可视化
        plt.figure(figsize=(12, 10))
        plt.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title(f'混淆矩阵（随机种子：{args.random_seed} | 三路径混合模型）', fontsize=14)
        plt.colorbar()
        tick_marks = np.arange(len(label_encoder.classes_))
        plt.xticks(tick_marks, label_encoder.classes_, rotation=45, ha='right')
        plt.yticks(tick_marks, label_encoder.classes_)

        thresh = conf_matrix.max() / 2.
        for i in range(conf_matrix.shape[0]):
            for j in range(conf_matrix.shape[1]):
                plt.text(j, i, format(conf_matrix[i, j], 'd'),
                         horizontalalignment="center",
                         color="white" if conf_matrix[i, j] > thresh else "black")

        plt.ylabel('真实标签', fontsize=12)
        plt.xlabel('预测标签', fontsize=12)
        plt.tight_layout()
        conf_plot_path = os.path.join(args.save_dir, f'confusion_matrix_seed_{args.random_seed}.png')
        plt.savefig(conf_plot_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"📊 混淆矩阵图已保存至：{conf_plot_path}")

        # 保存测试集混淆矩阵
        test_conf_matrix = test_eval['confusion_matrix']
        test_conf_df = pd.DataFrame(test_conf_matrix, index=label_encoder.classes_, columns=label_encoder.classes_)
        test_conf_csv_path = os.path.join(args.save_dir, f'test_confusion_matrix_seed_{args.random_seed}.csv')
        test_conf_df.to_csv(test_conf_csv_path, encoding='utf-8-sig')
        print(f"📊 测试集混淆矩阵已保存至：{test_conf_csv_path}")

        # 测试集混淆矩阵可视化
        plt.figure(figsize=(12, 10))
        plt.imshow(test_conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title(f'测试集混淆矩阵（随机种子：{args.random_seed} | 三路径混合模型）', fontsize=14)
        plt.colorbar()
        tick_marks = np.arange(len(label_encoder.classes_))
        plt.xticks(tick_marks, label_encoder.classes_, rotation=45, ha='right')
        plt.yticks(tick_marks, label_encoder.classes_)

        test_thresh = test_conf_matrix.max() / 2.
        for i in range(test_conf_matrix.shape[0]):
            for j in range(test_conf_matrix.shape[1]):
                plt.text(j, i, format(test_conf_matrix[i, j], 'd'),
                         horizontalalignment="center",
                         color="white" if test_conf_matrix[i, j] > test_thresh else "black")

        plt.ylabel('真实标签', fontsize=12)
        plt.xlabel('预测标签', fontsize=12)
        plt.tight_layout()
        test_conf_plot_path = os.path.join(args.save_dir, f'test_confusion_matrix_seed_{args.random_seed}.png')
        plt.savefig(test_conf_plot_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"📊 测试集混淆矩阵图已保存至：{test_conf_plot_path}")

        # 保存训练结果CSV
        csv_save_path = os.path.join(args.save_dir, args.save_csv_name + '.csv')
        save_cls_result(args, final_eval, total_train_time, label_encoder, csv_save_path, args.random_seed)

        # 训练历史图
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
        epochs = len(history['train_loss'])

        ax1.plot(range(1, epochs + 1), history['train_loss'], label='训练损失', color='#1f77b4')
        ax1.plot(range(1, epochs + 1), history['val_loss'], label='验证损失', color='#ff7f0e')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('损失值')
        ax1.set_title(f'损失曲线（种子：{args.random_seed} | 三路径混合模型）')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(range(1, epochs + 1), history['train_acc'], label='训练准确率', color='#1f77b4')
        ax2.plot(range(1, epochs + 1), history['val_acc'], label='验证准确率', color='#ff7f0e')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('准确率')
        ax2.set_title(f'准确率曲线（最佳验证准确率：{best_val_acc:.4f}）')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        ax3.plot(range(1, epochs + 1), history['lr'], label='学习率', color='#2ca02c')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('学习率')
        ax3.set_title('学习率变化曲线（余弦退火）')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        ax4.axis('off')
        config_text = f"""
        模型配置（随机种子：{args.random_seed} | 三路径混合模型）
        ========================
        数据目录：{args.data_dir}
        训练井{len(train_wells)}口 | 验证井{len(val_wells)}口 | 测试井{len(test_wells)}口
        数据维度：{args.num_channels}变量 × {args.seq_len}长度
        模型参数：{total_params / 1e6:.2f}M 可训练参数
        最佳结果：验证准确率={best_val_acc:.4f} | 测试准确率={test_eval['overall_accuracy']:.4f}
        训练时间：{total_train_time:.2f}秒
        模型架构：SoftShape MoE + 类别特定Transformer + 通用Transformer
        """
        ax4.text(0.1, 0.9, config_text, transform=ax4.transAxes, fontsize=10,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        plot_save_path = os.path.join(
            args.save_dir,
            f"training_history_"
            f"{args.experiment_name}_"
            f"seed_{args.random_seed}.png"
        )
        plt.tight_layout()
        plt.savefig(plot_save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"📈 训练历史图已保存至：{plot_save_path}")

        # 保存每口验证集井的结果表格
        print(f"\n=== 保存验证集每口井结果表格 ===")
        for well_name, well_info in final_eval['well_accuracy'].items():
            samples_df = well_info['sorted_samples'].copy()
            samples_df['true_label_name'] = samples_df['true'].map(lambda x: label_encoder.classes_[x])
            samples_df['pred_label_name'] = samples_df['pred'].map(lambda x: label_encoder.classes_[x])
            samples_df['is_correct'] = samples_df['true'] == samples_df['pred']

            save_cols = ['sample_id', 'true_label_name', 'pred_label_name', 'is_correct']
            well_result_df = samples_df[save_cols]

            well_result_path = os.path.join(args.save_dir,
                                            f'well_{well_name}_validation_result_seed_{args.random_seed}.csv')
            well_result_df.to_csv(well_result_path, index=False, encoding='utf-8-sig')
            print(f"📋 井 {well_name} 验证结果已保存至：{well_result_path}")

        # 保存每口测试集井的结果表格
        print(f"\n=== 保存测试集每口井结果表格 ===")
        for well_name, well_info in test_eval['well_accuracy'].items():
            samples_df = well_info['sorted_samples'].copy()
            samples_df['true_label_name'] = samples_df['true'].map(lambda x: label_encoder.classes_[x])
            samples_df['pred_label_name'] = samples_df['pred'].map(lambda x: label_encoder.classes_[x])
            samples_df['is_correct'] = samples_df['true'] == samples_df['pred']

            save_cols = ['sample_id', 'true_label_name', 'pred_label_name', 'is_correct']
            well_result_df = samples_df[save_cols]

            well_result_path = os.path.join(args.save_dir,
                                            f'well_{well_name}_test_result_seed_{args.random_seed}.csv')
            well_result_df.to_csv(well_result_path, index=False, encoding='utf-8-sig')
            print(f"📋 井 {well_name} 测试结果已保存至：{well_result_path}")

    print("\n=== 所有流程完成 ===")
